# М20. LLM целиком: сквозной расчёт

От токенов до вероятностей, производных, обновления весов и генерации. Это искусственный пример с 50 параметрами, а не языковая модель, обученная на корпусе. Все данные и код находятся внутри тетради. GPU, API, внешние файлы и загрузка весов не нужны; установка требуется только для отсутствующих библиотек.

Первый проход: шаги 1–3, 6–8 и 10–11. Второй: добавьте производные и оптимизатор в шагах 4–5, маршрутизацию в шаге 9. Выполняйте шаги выбранного маршрута по порядку. Числа лекции относятся к исходным матрицам. Для своих опытов меняйте температуру и top-k в отдельной ячейке шага 8, а входы и цели — в шаге 11: там результаты пересчитываются без требования совпасть с исходным эталоном.

In [ ]:
import importlib.util
import subprocess
import sys
missing = [name for name in ("numpy", "torch", "matplotlib") if importlib.util.find_spec(name) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing],
                          creationflags=getattr(subprocess, "CREATE_NO_WINDOW", 0))
import numpy as np
import torch
import matplotlib
from IPython import get_ipython
ipython = get_ipython()
if ipython is not None and getattr(ipython, "kernel", None) is not None:
    ipython.run_line_magic("matplotlib", "inline")
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(20)
np.set_printoptions(precision=6, suppress=True)
print("NumPy", np.__version__, "PyTorch", torch.__version__, "device: CPU")

## 1. Полный вычислительный путь

Следующая ячейка определяет функции, но ещё не обучает модель. `forward` возвращает и выход, и промежуточные значения. Формы матриц соответствуют лекции: строки — позиции, линейные веса хранятся как «вход × выход». `torch_forward` написана отдельно и использует float64. Численная производная будет вычисляться через NumPy, без графа PyTorch.

In [ ]:
"""M20: one complete pre-norm decoder block, with inspectable intermediates.

Rows are token representations; linear weights have shape [input, output].
This synthetic fixture illustrates arithmetic, not language competence.
"""
import numpy as np

EPS = 1e-6
INPUT = np.array([0, 1, 2], dtype=np.int64)
TARGET = np.array([1, 2, 3], dtype=np.int64)
TOKENS = ("BOS", "A", "B", "EOS")


def fixture():
    """Return fresh arrays: vocabulary 4, width 2, one head, FFN width 2."""
    return {
        "E": np.array([[1., 0.], [0., 1.], [1., 1.], [-1., 0.]]),
        "Q": .5 * np.eye(2), "K": .5 * np.eye(2),
        "V": np.diag([.5, 1.]), "O": .25 * np.eye(2),
        "gate": np.array([[.5, -.25], [.25, .5]]),
        "up": np.array([[1., .5], [-.5, 1.]]),
        "down": .25 * np.eye(2),
        "na": np.ones(2), "nf": np.ones(2), "nout": np.ones(2),
        "U": np.array([[0., .5, -.5, .25], [0., -.25, .5, -.5]]),
    }


def rms(x, gain):
    return x / np.sqrt(np.mean(x * x, axis=-1, keepdims=True) + EPS) * gain


def softmax(x):
    e = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)


def silu(x):
    return x / (1 + np.exp(-x))


def rope(x, positions):
    """[heads, time, even head_dim]; pair first and second halves."""
    dh = x.shape[-1]
    if dh % 2:
        raise ValueError("RoPE requires an even head dimension")
    angle = np.asarray(positions)[:, None] * 10000. ** (-np.arange(0, dh, 2) / dh)
    angle = np.concatenate([angle, angle], axis=-1)
    rotated = np.concatenate([-x[..., dh // 2:], x[..., :dh // 2]], axis=-1)
    return x * np.cos(angle) + rotated * np.sin(angle)


def forward(w, ids, hq=1, hkv=1, cache=None):
    """A single layer; cache contains already rotated K and unrotated V.

    Cache is valid only for these weights and this uninterrupted prefix.
    Tied embeddings are selected by omitting U from w.
    """
    ids = np.asarray(ids, dtype=np.int64)
    if ids.ndim != 1 or len(ids) == 0:
        raise ValueError("ids must be a nonempty one-dimensional sequence")
    if np.any(ids < 0) or np.any(ids >= len(w["E"])):
        raise ValueError("token id outside the vocabulary")
    h = w["E"][ids]
    t, d = h.shape
    if hq < 1 or hkv < 1 or hq % hkv or d % hq:
        raise ValueError("head counts must divide the query count and model width")
    dh = d // hq
    offset = 0 if cache is None else cache[0].shape[1]
    positions = np.arange(offset, offset + t)
    x = rms(h, w["na"])
    q = rope((x @ w["Q"]).reshape(t, hq, dh).transpose(1, 0, 2), positions)
    k = rope((x @ w["K"]).reshape(t, hkv, dh).transpose(1, 0, 2), positions)
    v = (x @ w["V"]).reshape(t, hkv, dh).transpose(1, 0, 2)
    if cache is not None:
        k = np.concatenate([cache[0], k], axis=1)
        v = np.concatenate([cache[1], v], axis=1)
    kr, vr = (np.repeat(a, hq // hkv, axis=0) for a in (k, v))
    scores = q @ kr.transpose(0, 2, 1) / np.sqrt(dh)
    mask = np.arange(k.shape[1])[None, :] <= positions[:, None]
    a = softmax(np.where(mask[None], scores, -np.inf))
    mixed = (a @ vr).transpose(1, 0, 2).reshape(t, d)
    z = h + mixed @ w["O"]
    n = rms(z, w["nf"])
    gate, up = n @ w["gate"], n @ w["up"]
    f = (silu(gate) * up) @ w["down"]
    h1 = z + f
    final = rms(h1, w["nout"])
    logits = final @ (w["U"] if "U" in w else w["E"].T)
    return dict(H0=h, X=x, Q=q, K=k, V=v, scores=scores, A=a,
                mixed=mixed, Z=z, N=n, gate=gate, up=up, F=f, H1=h1,
                final=final, logits=logits, probabilities=softmax(logits), cache=(k, v))


def cross_entropy(logits, targets, mask=None):
    """Mean over explicitly selected targets; never mean over padding."""
    targets = np.asarray(targets, dtype=np.int64)
    if logits.ndim != 2 or targets.shape != (len(logits),):
        raise ValueError("one target is required for every logit row")
    selected = np.ones(len(targets), dtype=bool) if mask is None else np.asarray(mask, dtype=bool)
    if selected.shape != targets.shape or not selected.any():
        raise ValueError("the target mask must select at least one row")
    if np.any(targets[selected] < 0) or np.any(targets[selected] >= logits.shape[1]):
        raise ValueError("selected target outside the vocabulary")
    x = logits[selected]
    m = x.max(axis=-1)
    return np.mean(m + np.log(np.exp(x - m[:, None]).sum(axis=-1))
                   - x[np.arange(len(x)), targets[selected]])


def finite_gradients(w, ids=INPUT, targets=TARGET, step=1e-5):
    """Central differences for every scalar; restore each weight after use."""
    result = {}
    for name, value in w.items():
        g = np.empty_like(value)
        for index in np.ndindex(value.shape):
            old = value[index]
            try:
                value[index] = old + step
                plus = cross_entropy(forward(w, ids)["logits"], targets)
                value[index] = old - step
                minus = cross_entropy(forward(w, ids)["logits"], targets)
            finally:
                value[index] = old
            g[index] = (plus - minus) / (2 * step)
        result[name] = g
    return result


def adamw_first_step(w, grads, lr=.01, decay=.1, clip=1., eps=1e-8):
    """Zero initial moments, bias correction: m_hat=g, v_hat=g^2.

    Only matrices receive weight decay. Clipping precedes moment updates.
    The first bias-corrected step is independent of beta1 and beta2.
    """
    norm = np.sqrt(sum(np.sum(g * g) for g in grads.values()))
    scale = min(1., clip / max(norm, 1e-300))
    updated = {}
    for name, value in w.items():
        g = grads[name] * scale
        wd = decay if value.ndim == 2 else 0.
        updated[name] = (1 - lr * wd) * value - lr * g / (np.abs(g) + eps)
    return updated, float(norm), float(scale)


def torch_forward(w, ids, hq=1, hkv=1):
    """Independent float64 autograd path; PyTorch is imported only when used."""
    import torch
    import torch.nn.functional as F
    ids = torch.as_tensor(ids, dtype=torch.long)
    h = w["E"][ids]
    t, d = h.shape
    dh = d // hq
    def norm(a, gain):
        return a * torch.rsqrt(a.square().mean(-1, keepdim=True) + EPS) * gain
    def rotate(a):
        freq = 10000. ** (-torch.arange(0, dh, 2, dtype=h.dtype) / dh)
        angle = torch.arange(t, dtype=h.dtype)[:, None] * freq
        angle = torch.cat([angle, angle], dim=-1)
        return a * angle.cos() + torch.cat([-a[..., dh//2:], a[..., :dh//2]], -1) * angle.sin()
    n = norm(h, w["na"])
    q = rotate((n @ w["Q"]).view(t, hq, dh).transpose(0, 1))
    k = rotate((n @ w["K"]).view(t, hkv, dh).transpose(0, 1))
    v = (n @ w["V"]).view(t, hkv, dh).transpose(0, 1)
    k, v = [a.repeat_interleave(hq // hkv, dim=0) for a in (k, v)]
    scores = q @ k.transpose(-2, -1) / dh ** .5
    scores = scores.masked_fill(torch.ones(t, t, dtype=torch.bool).triu(1), -torch.inf)
    mixed = (scores.softmax(-1) @ v).transpose(0, 1).reshape(t, d)
    z = h + mixed @ w["O"]
    n = norm(z, w["nf"])
    h1 = z + (F.silu(n @ w["gate"]) * (n @ w["up"])) @ w["down"]
    return norm(h1, w["nout"]) @ (w["U"] if "U" in w else w["E"].T)


def autograd_gradients(w, ids=INPUT, targets=TARGET):
    import torch
    params = {k: torch.tensor(v, dtype=torch.float64, requires_grad=True) for k, v in w.items()}
    logits = torch_forward(params, ids)
    loss = torch.nn.functional.cross_entropy(logits, torch.tensor(targets, dtype=torch.long))
    loss.backward()
    return {k: v.grad.detach().numpy().copy() for k, v in params.items()}


def sampling_distribution(logits, temperature=1., top_k=None):
    """One row; stable token-index tie rule. Distribution q, not base p."""
    if temperature <= 0 or np.asarray(logits).ndim != 1:
        raise ValueError("use a positive temperature and one logit row")
    z = np.asarray(logits) / temperature
    if top_k is not None:
        if not isinstance(top_k, int) or not 1 <= top_k <= len(z):
            raise ValueError("top_k must be an integer in [1, vocabulary_size]")
        keep = np.argsort(-z, kind="stable")[:top_k]
        z = np.where(np.isin(np.arange(len(z)), keep), z, -np.inf)
    return softmax(z)


def moe_weights(logits, selection_bias, k=2, eps=1e-9):
    """MiniLLM's sigmoid routing: bias changes selection, not mixture weights."""
    p = 1 / (1 + np.exp(-np.asarray(logits)))
    selected = np.argsort(-(p + selection_bias), kind="stable")[:k]
    return selected, p[selected] / (p[selected].sum() + eps)


def architecture_examples():
    """Small exact SSM recurrence and a two-step discrete reverse chain."""
    state, states = 0., []
    for u in (1., 0., -1.):
        state = .5 * state + u
        states.append(state)
    # Four binary sequences: 00, 01, 10, 11. At s=2 both positions are masked.
    first = np.array([.6*.3, .6*.7, .4*.3, .4*.7])
    # At the next refinement each bit is independently kept with probability .8.
    bits = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
    distance = (bits[:, None, :] != bits[None, :, :]).sum(-1)
    kernel = .8 ** (2-distance) * .2 ** distance
    return {"ssm_states": states, "diffusion_first": first.tolist(),
            "diffusion_kernel": kernel.tolist(), "diffusion_final": (first @ kernel).tolist()}


def reference_metrics():
    w = fixture()
    trace = forward(w, INPUT)
    grads = autograd_gradients(w)
    changed, norm, scale = adamw_first_step(w, grads)
    out = {k: v.tolist() for k, v in trace.items() if isinstance(v, np.ndarray)}
    out.update(parameter_count=sum(v.size for v in w.values()),
               loss=float(cross_entropy(trace["logits"], TARGET)),
               target_losses=(-np.log(trace["probabilities"][np.arange(3), TARGET])).tolist(),
               gradient_norm=norm, clip_scale=scale,
               gradients={k: v.tolist() for k, v in grads.items()},
               loss_after_step=float(cross_entropy(forward(changed, INPUT)["logits"], TARGET)),
               probabilities_after_step=forward(changed, INPUT)["probabilities"].tolist(),
               q_temperature_half_top2=sampling_distribution(trace["logits"][-1], .5, 2).tolist(),
               architecture_examples=architecture_examples())
    return out

In [ ]:
baseline = fixture()
trace = forward(baseline, INPUT)
assert sum(v.size for v in baseline.values()) == 50
for name in ("H0", "X", "Q", "K", "V", "A", "Z", "N", "gate", "up", "F", "H1", "final"):
    print(name, trace[name].shape, "\n", trace[name])

## 2. Внимание, логиты и вероятности

В каждой строке внимания сумма равна 1. Правее диагонали должны стоять нули. Проследите последнюю строку: смешиваются три значения, но первая строка использует только своё. Вероятности ниже относятся к BOS, A, B, EOS именно в таком порядке.

In [ ]:
np.testing.assert_allclose(trace["A"].sum(-1), 1.)
assert np.all(np.triu(trace["A"][0], 1) == 0)
print("logits\n", trace["logits"])
print("p\n", trace["probabilities"])
fig, ax = plt.subplots(figsize=(6, 4))
ax.imshow(trace["A"][0], vmin=0, vmax=1, cmap="Blues")
for i in range(3):
    for j in range(3):
        ax.text(j, i, f"{trace['A'][0,i,j]:.3f}", ha="center", va="center",
                color="white" if trace["A"][0,i,j] > .6 else "#2E3440")
ax.set(xticks=range(3), yticks=range(3), xlabel="Позиция ключа", ylabel="Позиция запроса", title="Причинное внимание")
fig.tight_layout()
plt.show()

## 3. Сдвиг целей и маска потери

Входы `[BOS, A, B]` предсказывают `[A, B, EOS]`. Сначала усредним все три потери, затем выберем только последнюю. Это меняет цель обучения, сохраняя logits. Маска не убирает токены из контекста. Пустой набор целей должен дать понятную ошибку, а не NaN.

In [ ]:
loss_before = cross_entropy(trace["logits"], TARGET)
loss_eos = cross_entropy(trace["logits"], TARGET, [False, False, True])
np.testing.assert_allclose(loss_before, 1.1093739242148548, atol=1e-10)
np.testing.assert_allclose(loss_eos, 1.6955297196508368, atol=1e-10)
print("all targets:", loss_before, "EOS only:", loss_eos)
try:
    cross_entropy(trace["logits"], TARGET, [False]*3)
except ValueError as error:
    print(type(error).__name__, str(error))
else:
    raise AssertionError("empty target mask was accepted")

## 4. Проверим все производные

Изменение одного параметра на маленькое положительное и отрицательное число даёт независимую оценку его производной. Проверим все 50 скаляров; потом уберём независимую U и повторим для 42 параметров связанного варианта. Это также проверяет суммирование градиентов двух ролей E. Слишком маленький шаг конечной разности может усилить ошибку округления; мы используем $10^{-5}$.

In [ ]:
gradients = autograd_gradients(baseline)
finite = finite_gradients(baseline)
max_error = max(np.max(np.abs(gradients[k] - finite[k])) for k in baseline)
assert max_error < 2e-8
print("50 parameters; max gradient error:", max_error)
tied = fixture()
del tied["U"]
tied_auto, tied_fd = autograd_gradients(tied), finite_gradients(tied)
tied_error = max(np.max(np.abs(tied_auto[k] - tied_fd[k])) for k in tied)
assert tied_error < 2e-8
print("42 tied parameters; max gradient error:", tied_error)
logit_gradient = trace["probabilities"].copy()
logit_gradient[np.arange(len(TARGET)), TARGET] -= 1
logit_gradient /= len(TARGET)
np.testing.assert_allclose(logit_gradient.sum(-1), 0., atol=1e-15)
print("gradient of loss w.r.t. logits:\n", logit_gradient)

## 5. Один шаг AdamW

Начальные моменты нулевые. Матрицы получают decay 0.1, gains нормализации — 0. Норма всего градиента меньше порога 1. Сравним формулу первого шага с настоящим оптимизатором, включая каждую обновлённую матрицу. Результат после шага относится к прежнему обучающему примеру, а не к качеству на новых данных.

In [ ]:
updated, gradient_norm, clip_scale = adamw_first_step(baseline, gradients)
params = {k: torch.tensor(v, dtype=torch.float64, requires_grad=True) for k, v in baseline.items()}
optimizer = torch.optim.AdamW([
    {"params": [p for p in params.values() if p.ndim == 2], "weight_decay": .1},
    {"params": [p for p in params.values() if p.ndim == 1], "weight_decay": 0.}],
    lr=.01, betas=(.9, .95), eps=1e-8)
loss_torch = torch.nn.functional.cross_entropy(torch_forward(params, INPUT), torch.tensor(TARGET))
optimizer.zero_grad()
loss_torch.backward()
for p in params.values():
    p.grad.mul_(clip_scale)
optimizer.step()
for k, p in params.items():
    np.testing.assert_allclose(p.detach().numpy(), updated[k], atol=1e-10)
loss_after = cross_entropy(forward(updated, INPUT)["logits"], TARGET)
np.testing.assert_allclose(loss_after, 1.0848315484318878, atol=1e-10)
print("gradient norm:", gradient_norm, "scale:", clip_scale)
print("loss:", loss_before, "->", loss_after)

## 6. Несколько шагов: состояние оптимизатора сохраняется

Здесь создаётся новый оптимизатор, после чего 80 шагов выполняются последовательно. Это намеренное переобучение на одной строке. Мы проверяем работоспособность обучения, а не способность понимать язык. Отличайте эту кривую от предыдущего одиночного шага: здесь скорость обучения равна 0.03, а weight decay отключён.

In [ ]:
training_params = {k: torch.tensor(v, dtype=torch.float64, requires_grad=True) for k, v in fixture().items()}
training_optimizer = torch.optim.AdamW(list(training_params.values()), lr=.03, weight_decay=0.)
training_losses = []
for step in range(80):
    training_optimizer.zero_grad()
    loss = torch.nn.functional.cross_entropy(torch_forward(training_params, INPUT), torch.tensor(TARGET))
    training_losses.append(loss.item())
    loss.backward()
    torch.nn.utils.clip_grad_norm_(list(training_params.values()), 1.)
    training_optimizer.step()
with torch.no_grad():
    final_training_loss = torch.nn.functional.cross_entropy(torch_forward(training_params, INPUT), torch.tensor(TARGET)).item()
training_losses.append(final_training_loss)
assert np.isfinite(training_losses).all() and final_training_loss < training_losses[0]
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(training_losses, color="#5E81AC")
ax.set(xlabel="Шаг обучения", ylabel="Средняя потеря, нат", title="Обучение на одной последовательности")
fig.tight_layout()
plt.show()
print("training loss:", training_losses[0], "->", final_training_loss)

## 7. Причинность и кэш

Изменим последний вход: более ранние logits должны сохраниться. Затем вычислим исходный префикс по одному токену, передавая K/V между вызовами. Отдельно проверим два новых токена за один вызов. Весам baseline этот кэш соответствует; после обучения он уже непригоден. Если забыть абсолютную позицию RoPE, изменятся оценки внимания — последняя проверка специально показывает этот контрпример.

In [ ]:
changed_input = forward(baseline, [0, 1, 3])
np.testing.assert_allclose(changed_input["logits"][:2], trace["logits"][:2])
cache, cached_logits = None, []
for token in INPUT:
    current = forward(baseline, [token], cache=cache)
    cached_logits.append(current["logits"])
    cache = current["cache"]
np.testing.assert_allclose(np.concatenate(cached_logits), trace["logits"], atol=1e-12)
prefix = forward(baseline, INPUT[:1])
suffix = forward(baseline, INPUT[1:], cache=prefix["cache"])
np.testing.assert_allclose(suffix["logits"], trace["logits"][1:], atol=1e-12)
wrong_q = rope((trace["X"][2:] @ baseline["Q"])[None], [0])
wrong_scores = wrong_q @ trace["K"].transpose(0, 2, 1) / np.sqrt(2)
offset_error = np.max(np.abs(wrong_scores[0, 0] - trace["scores"][0, 2]))
assert offset_error > .1
print("cache scalars:", sum(a.size for a in cache), "wrong-offset score difference:", offset_error)

## 8. Распределение декодера и короткая генерация

Сначала повторим top-2 и температуру 0.5 для исходной модели. Затем сгенерируем ответ моделью после короткого обучения, жадно выбирая токены. Это две разные проверки: первая относится к таблице лекции, вторая — к полученному чекпойнту. Максимум длины обеспечивает остановку даже при отсутствии EOS.

In [ ]:
q = sampling_distribution(trace["logits"][-1], temperature=.5, top_k=2)
np.testing.assert_allclose(q, [0., .5631163013756344, .4368836986243656, 0.])
print("p:", trace["probabilities"][-1], "q:", q)
trained = {k: p.detach().numpy().copy() for k, p in training_params.items()}
generated = [0]
generation_cache = None
for _ in range(12):
    output = forward(trained, [generated[-1]], cache=generation_cache)
    generation_cache = output["cache"]
    next_token = int(np.argmax(output["logits"][-1]))
    generated.append(next_token)
    if next_token == 3:
        break
print(" ".join(TOKENS[i] for i in generated))

### Ваш опыт: температура и top-k

Вернёмся к исходным весам и последнему префиксу `[BOS, A, B]`. Следующая ячейка берёт его логиты из `trace`: короткое обучение выше их не меняло. Задайте `experiment_temperature` положительным конечным числом; `experiment_top_k` может быть целым числом от 1 до 4 или `None`, если нужно оставить весь словарь. Здесь проверяются допустимость вероятностей и сумма 1, поэтому новая температура не обязана давать прежние числа.

Сначала сравните температуры 0.5, 1.0 и 2.0 при top-2. Как меняется соотношение A и B? Затем установите температуру 1.0 и `None`: распределение выбора должно совпасть с исходным распределением модели. Объясните, почему при top-2 у EOS по-прежнему нулевая вероятность выбора, хотя его вероятность в модели положительна. В этом опыте меняется правило выбора, а веса и логиты остаются фиксированными.

In [ ]:
experiment_temperature = 1.0
experiment_top_k = 2
if not np.isfinite(experiment_temperature) or experiment_temperature <= 0:
    raise ValueError("Температура должна быть положительным конечным числом")
experiment_q = sampling_distribution(
    trace["logits"][-1], temperature=experiment_temperature, top_k=experiment_top_k)
assert np.isfinite(experiment_q).all() and (experiment_q >= 0).all()
np.testing.assert_allclose(experiment_q.sum(), 1., rtol=1e-12, atol=1e-12)
print("Температура:", experiment_temperature, "top-k:", experiment_top_k)
print("Токен   p модели   q выбора")
for token, model_p, selection_q in zip(TOKENS, trace["probabilities"][-1], experiment_q):
    print(f"{token:<5}   {model_p:.6f}   {selection_q:.6f}")

## 9. Маршрутизация MoE: выбор и вес — разные операции

Балансировочная поправка выбирает экспертов, но вес смеси вычисляется из исходных sigmoid-оценок. Ячейка воспроизводит только маршрутизацию из §15; это не второй обученный MoE. Чтобы получить полный подслой, нужно умножить каждый вес на выход соответствующего эксперта и добавить общих экспертов.

In [ ]:
selected, routing_weights = moe_weights([0., 1., -1.], np.array([0., -.5, .5]), k=2)
assert selected.tolist() == [2, 0]
np.testing.assert_allclose(routing_weights, [.3497554090542189, .6502445909457811])
print("selected:", selected, "weights:", routing_weights)

## 10. SSM и сумма скрытых траекторий

Скалярная SSM переносит состояние между входными позициями. Вторая задача суммирует пути двухшаговой обратной цепи над целыми бинарными строками. Её переходы заданы вручную, а не получены обучением диффузионной модели. Проверьте вероятность 00 скалярной суммой, затем объясните различие индексов времени в двух опытах.

In [ ]:
examples = architecture_examples()
np.testing.assert_allclose(examples["ssm_states"], [1., .5, -.75])
kernel = np.array(examples["diffusion_kernel"])
first = np.array(examples["diffusion_first"])
np.testing.assert_allclose(kernel.sum(-1), 1.)
final = first @ kernel
np.testing.assert_allclose(final, [.2128, .3472, .1672, .2728])
manual_00 = .18*.64 + .42*.16 + .12*.16 + .28*.04
np.testing.assert_allclose(manual_00, final[0])
print("SSM:", examples["ssm_states"])
print("reverse chain:", first, "->", final, "sum:", final.sum())

## 11. Ваш опыт: изменяем вход или цель

Можно менять длину непустого входа и идентификаторы от 0 до 3; число целей должно совпадать с числом входных позиций. Для последовательности `[0, 2, 1, 3]` задайте вход `[0, 2, 1]`, цели `[2, 1, 3]`. Цели могут проверять и альтернативное продолжение того же входа. Результаты здесь пересчитываются, а не сравниваются с фиксированной потерей лекции.

До запуска предскажите: какие строки изменятся, будет ли EOS более вероятен, как изменится потеря? После запуска объясните расхождение с прогнозом. Не переиспользуйте кэш от другого входа.

In [ ]:
experiment_ids = np.array([0, 2, 1], dtype=np.int64)
experiment_targets = np.array([2, 1, 3], dtype=np.int64)
experiment_weights = fixture()
experiment_trace = forward(experiment_weights, experiment_ids)
experiment_loss = cross_entropy(experiment_trace["logits"], experiment_targets)
experiment_gradients = autograd_gradients(experiment_weights, experiment_ids, experiment_targets)
print("inputs:", experiment_ids, "targets:", experiment_targets)
print("p:\n", experiment_trace["probabilities"])
print("loss:", experiment_loss)
print("gradient norm:", np.sqrt(sum(np.sum(g*g) for g in experiment_gradients.values())))

### Что удалось проверить

Мы прошли весь путь от lookup до потери и обновления, проверили все производные, связанный выход, причинность и эквивалентность кэша. Также отделили выбор токена от предсказания и повторили арифметику двух альтернативных конструкций. На этом масштабе проверяется вычислительный механизм; качество языка требует данных и отдельной оценки, как в М19.